# Bloom Filter Encoding

## Objective
Convert identifiers into privacy-preserving representations using Bloom filters.

### Load data

In [ ]:
import pandas as pd
from bitarray import bitarray
import hashlib

# Load
df_A = pd.read_pickle("../data/processed/df_A_preprocessed.pkl")
df_B = pd.read_pickle("../data/processed/df_B_preprocessed.pkl")



### Q-grams

introduces error tolerance - they allow fuzzy matching

In [ ]:
def get_qgrams(text, q=2):
    if not isinstance(text, str):
        text = str(text) if text is not None else ""
    text = text.replace(" ", "")
    return [text[i:i+q] for i in range(len(text) - q + 1)]


get_qgrams("otienootieno2010-10-15")

['Da', 'an', 'ni', 'ie', 'el', 'lN', 'Nd', 'de', 'er', 'ri', 'it', 'tu']

Q-grams allow approximate matching by capturing substrings.

### Bloom Filter

In [ ]:
from bitarray import bitarray
import hashlib

def create_bloom_filter(text, size=1024, num_hashes=5):
    # Creating a blank memory
    bf = bitarray(size)
    bf.setall(0)

    qgrams = get_qgrams(text) # Breaking the word into pieces (overlapping pieces)

    for qg in qgrams:
        for i in range(num_hashes):  # Create multiple versions of the same qgram
            h = int(hashlib.sha1((qg + str(i)).encode()).hexdigest(), 16) # Turn each version into a number
            bf[h % size] = 1         # Pick a spot in the list and mark the spot

    return bf
    '''
    Combine: "DA" + "0" = "DA0", Encode: b'DA0' - byte (DAO), Hash: SHA1 gives us a fingerprint, 
    Hexdigest: "a94a8fe5ccb19ba61c4c0873d391e987982fbbd3", 
    Convert: This becomes the number 763275293847209384720938472093847209384720938
    '''
    # 

#### Apply encoding

"john1990" → 010010101010... -> what is shared instead of raw identifiers

In [22]:
df_A["bloom"] = df_A["combined"].apply(create_bloom_filter)
df_B["bloom"] = df_B["combined"].apply(create_bloom_filter)

Each record is now represented as a bit array instead of raw identifiers.

### Save


In [23]:
df_A.to_pickle("../data/processed/df_A_encoded.pkl")
df_B.to_pickle("../data/processed/df_B_encoded.pkl")